# 07 — Reset this project's experiment data
This notebook is for a fresh B1 run. It clears the application RAW/CLEAN/ingestion/inference tables, the `diabetes-readmission` registered model and active runs/logged models from the named diabetes experiments. It deletes **all objects and both buckets** `diabetes-source` and `mlflow-artifacts`. It does not uninstall Kubernetes services or delete the local source CSV.

MLflow run deletion is soft: old runs may remain in MLflow's deleted history. Their artifact objects are removed with the bucket, so those artifacts cannot be recovered from MLflow. Run notebook 01 again afterwards to recreate the buckets and upload the source batches. Stop pipeline runs before resetting. The reset requires typing the confirmation in a later cell.

If Jupyter has restarted, first run **only the dependency installation cell** in notebook 01 and restart the kernel. Do not upload batches until after the reset.


In [1]:
import os
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from mlflow.exceptions import MlflowException
from botocore.exceptions import ClientError
from sqlalchemy import create_engine, text
from IPython.display import display

## Connection functions
These cells are copies of the connection functions from notebook 05.

In [2]:
import os
from urllib.parse import quote_plus


In [3]:
def database_url():
    user = quote_plus(os.environ['DIABETES_DB_USER'])
    pwd = quote_plus(os.environ['DIABETES_DB_PASSWORD'])
    host = os.getenv('DIABETES_DB_HOST', 'diabetes-postgres.mlops-project.svc.cluster.local')
    db = quote_plus(os.getenv('DIABETES_DB_NAME', 'diabetes'))
    return f'postgresql+psycopg2://{user}:{pwd}@{host}:5432/{db}'


In [4]:
def s3_client():
    import boto3
    return boto3.client('s3', endpoint_url=os.getenv('S3_ENDPOINT_URL','http://minio.mlops-project.svc.cluster.local:9000'), region_name='us-east-1', aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'], aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'])


In [5]:
PROJECT_EXPERIMENTS = ('diabetes-first-batch', 'diabetes-pipeline', 'diabetes-development')
REGISTERED_MODEL = 'diabetes-readmission'
PROJECT_BUCKETS = ('diabetes-source', 'mlflow-artifacts')
DB_TABLES = ('clean.encounters', 'raw.encounters', 'audit.ingestion_batches', 'audit.inference_logs')
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
client = MlflowClient()
s3 = s3_client()
db = create_engine(database_url(), pool_pre_ping=True)

## Preview the scope without deleting anything
Counts are informational. Bucket counts include every object in each named bucket.

In [6]:
with db.connect() as connection:
    table_counts = []
    for name in DB_TABLES:
        exists = connection.execute(text('SELECT to_regclass(:name)'), {'name': name}).scalar() is not None
        count = connection.execute(text('SELECT count(*) FROM ' + name)).scalar() if exists else None
        table_counts.append({'table': name, 'rows': count})
display(pd.DataFrame(table_counts))
experiments = [e for e in client.search_experiments(view_type=ViewType.ALL) if e.name in PROJECT_EXPERIMENTS]
display(pd.DataFrame([{'experiment': e.name, 'id': e.experiment_id, 'status': e.lifecycle_stage} for e in experiments]))
versions = list(client.search_model_versions(f"name='{REGISTERED_MODEL}'"))
print('Registered model versions:', [v.version for v in versions])
for bucket in PROJECT_BUCKETS:
    try:
        s3.head_bucket(Bucket=bucket)
        count = sum(len(page.get('Contents', [])) for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket))
        print(f'{bucket}: {count} current objects')
    except ClientError as exc:
        if exc.response['Error']['Code'] in ('404', 'NoSuchBucket', 'NotFound'):
            print(f'{bucket}: absent')
        else:
            raise

,table,rows
0,clean.encounters,15000
1,raw.encounters,15000
2,audit.ingestion_batches,1
3,audit.inference_logs,0


,experiment,id,status
0,diabetes-pipeline,3,active
1,diabetes-first-batch,2,active
2,diabetes-development,1,deleted


Registered model versions: ['2', '1']
diabetes-source: 7 current objects
mlflow-artifacts: 24 current objects


## Reset functions
These functions operate only on the table names, experiment names, registered model and buckets listed above.

In [7]:
def collect_pages(fetch):
    items, token = [], None
    while True:
        page = fetch(token)
        items.extend(page)
        token = getattr(page, 'token', None)
        if not token:
            return items

In [8]:
def reset_mlflow(client):
    try:
        client.get_registered_model(REGISTERED_MODEL)
    except MlflowException as exc:
        if exc.error_code != 'RESOURCE_DOES_NOT_EXIST':
            raise
    else:
        client.delete_registered_model(REGISTERED_MODEL)
        print('Deleted registered model:', REGISTERED_MODEL)
    experiments = collect_pages(lambda token: client.search_experiments(view_type=ViewType.ALL, max_results=1000, page_token=token))
    for exp in experiments:
        if exp.name not in PROJECT_EXPERIMENTS or exp.lifecycle_stage != 'active':
            continue
        runs = collect_pages(lambda token: client.search_runs([exp.experiment_id], run_view_type=ViewType.ACTIVE_ONLY, max_results=1000, page_token=token))
        models = collect_pages(lambda token: client.search_logged_models([exp.experiment_id], max_results=1000, page_token=token))
        for model in models:
            client.delete_logged_model(model.model_id)
        for run in runs:
            client.delete_run(run.info.run_id)
        print(f'{exp.name}: deleted {len(runs)} active runs and {len(models)} logged models; experiment name retained')

In [9]:
def reset_database(db):
    with db.begin() as connection:
        existing = [name for name in DB_TABLES
                    if connection.execute(text('SELECT to_regclass(:name)'), {'name': name}).scalar() is not None]
        if existing:
            connection.execute(text('TRUNCATE TABLE ' + ', '.join(existing)))
    print('Cleared application tables:', existing)

In [10]:
def delete_project_bucket(s3, bucket):
    if bucket not in PROJECT_BUCKETS:
        raise ValueError('Bucket is outside the project reset scope')
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError as exc:
        if exc.response['Error']['Code'] in ('404', 'NoSuchBucket', 'NotFound'):
            print('Already absent:', bucket)
            return
        raise
    status = s3.get_bucket_versioning(Bucket=bucket).get('Status')
    if status in ('Enabled', 'Suspended'):
        while True:
            page = s3.list_object_versions(Bucket=bucket, MaxKeys=1000)
            objects = [{'Key': x['Key'], 'VersionId': x['VersionId']}
                       for x in page.get('Versions', []) + page.get('DeleteMarkers', [])]
            if not objects: break
            result = s3.delete_objects(Bucket=bucket, Delete={'Objects': objects, 'Quiet': True})
            if result.get('Errors'): raise RuntimeError(result['Errors'])
    while True:
        page = s3.list_objects_v2(Bucket=bucket, MaxKeys=1000)
        objects = [{'Key': x['Key']} for x in page.get('Contents', [])]
        if not objects: break
        result = s3.delete_objects(Bucket=bucket, Delete={'Objects': objects, 'Quiet': True})
        if result.get('Errors'): raise RuntimeError(result['Errors'])
    s3.delete_bucket(Bucket=bucket)
    print('Deleted bucket and objects:', bucket)

## Confirm and execute
Review the preview and stop any Airflow or notebook pipeline run. To perform the reset, set the value below to `RESET PROJECT 2`, then run the following cell. This permanently deletes the objects in the two named buckets and removes the registered model.

In [11]:
CONFIRM_RESET = 'RESET PROJECT 2'  # Set to 'RESET PROJECT 2' only when you want to erase the project data.

In [12]:
if CONFIRM_RESET != 'RESET PROJECT 2':
    raise ValueError('Reset not executed. Review the scope and set CONFIRM_RESET first.')
reset_mlflow(client)
reset_database(db)
for bucket in PROJECT_BUCKETS:
    delete_project_bucket(s3, bucket)
print('Project reset complete. Run notebook 01 before starting B1 again.')

Deleted registered model: diabetes-readmission
diabetes-pipeline: deleted 2 active runs and 2 logged models; experiment name retained
diabetes-first-batch: deleted 2 active runs and 2 logged models; experiment name retained
Cleared application tables: ['clean.encounters', 'raw.encounters', 'audit.ingestion_batches', 'audit.inference_logs']
Deleted bucket and objects: diabetes-source
Deleted bucket and objects: mlflow-artifacts
Project reset complete. Run notebook 01 before starting B1 again.


If a reset step fails, the earlier steps may already have completed. Inspect the error and run the reset again; the functions handle already-absent resources. Run notebook 01 afterwards to recreate the buckets.